# Pivot queries

In [1]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd

In [2]:
# load_dotenv()
load_dotenv(dotenv_path=".env") 

conn = mysql.connector.connect(
    user = os.getenv("MYSQL_USER"),
    host = os.getenv("MYSQL_HOST"),
    port = os.getenv("MYSQL_PORT"),
    password = os.getenv("MYSQL_PASSWORD"),
    database = "recursive_cte_db"
)

print("Connected to the MySQL server successfully")

Connected to the MySQL server successfully


In [3]:
cursor = conn.cursor()

In [4]:
query = """
create table if not exists product_sales_tbl (
    agent varchar(50),
    country varchar(50),
    sales_amount int,
    unique (agent, country, sales_amount)
)
"""
cursor.execute(query)

query = """
insert into product_sales_tbl (agent, country, sales_amount)
values
('tom', 'uk', 200),
('john', 'us', 180),
('john', 'uk', 260),
('david', 'india', 450),
('tom', 'india', 350),
('david', 'us', 200),
('tom', 'us', 130),
('john', 'india', 540),
('john', 'uk', 120),
('david', 'uk', 220),
('john', 'uk', 420),
('david', 'us', 320),
('tom', 'us', 340),
('tom', 'uk', 660),
('john', 'india', 430),
('david', 'india', 230),
('david', 'india', 280),
('tom', 'uk', 480),
('john', 'us', 360),
('david', 'uk', 140);
"""
cursor.execute(query)

In [5]:
cursor.execute("select * from product_sales_tbl")
product_sales = cursor.fetchall()
headers = [tup[0] for tup in cursor.description]
pd.DataFrame(product_sales, columns=headers)

,agent,country,sales_amount
0,david,india,230
1,david,india,280
2,david,india,450
3,david,uk,140
4,david,uk,220
5,david,us,200
6,david,us,320
7,john,india,430
8,john,india,540
9,john,uk,120


## GROUP BY query

In [6]:
query = """
select country, agent, sum(sales_amount) as tot_sales
from product_sales_tbl
group by country, agent
order by country, agent
"""
cursor.execute(query)
country_agent_sales = cursor.fetchall()
headers = [tup[0] for tup in cursor.description]
pd.DataFrame(country_agent_sales, columns=headers)

,country,agent,tot_sales
0,india,david,960
1,india,john,970
2,india,tom,350
3,uk,david,360
4,uk,john,800
5,uk,tom,1340
6,us,david,520
7,us,john,540
8,us,tom,470


## pivot query using sql's `case when` 

In [7]:
query = """
select 
    agent,
    sum(case when country = 'india' then sales_amount else 0 end) as india,
    sum(case when country = 'uk' then sales_amount else 0 end) as uk,
    sum(case when country = 'us' then sales_amount else 0 end) as us
from product_sales_tbl
group by agent
order by agent
"""
cursor.execute(query)

pivot_data = cursor.fetchall()
headers = [tup[0] for tup in cursor.description]
display(pd.DataFrame(pivot_data, columns=headers))

,agent,india,uk,us
0,david,960,360,520
1,john,970,800,540
2,tom,350,1340,470


## Pivot query using Pandas

In [8]:
cursor.execute("select * from product_sales_tbl")
product_sales = cursor.fetchall()
headers = [tup[0] for tup in cursor.description]

In [9]:
df = pd.DataFrame(product_sales, columns=headers)

In [10]:
pivot_df = df.pivot_table(
    index='agent',
    columns='country',
    values='sales_amount',
    fill_value=0,
    aggfunc='sum'
).infer_objects(copy=False)

display(pivot_df)

country,india,uk,us
agent,,,
david,960,360,520
john,970,800,540
tom,350,1340,470
